# 01 — Analyse exploratoire des données (Phase 1)

Correspond à la Phase 1 de `docs/WORKFLOW.md`. Peut se lancer indépendamment de `00_setup_environment.ipynb` (pas besoin de GPU ni du modèle complet — seulement le tokenizer).

## 1 — Récupérer le dépôt et charger les données

In [ ]:
import os

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

DATA_DIR = f'{REPO_DIR}/data'

In [ ]:
!pip install -q -U pandas matplotlib transformers sentencepiece

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

print('Train:', train.shape, '| Val:', val.shape, '| Test:', test.shape)
train.head(3)

## 2 — Types, valeurs manquantes, doublons d'ID

In [ ]:
for name, df in [('Train', train), ('Val', val), ('Test', test)]:
    print(f'--- {name} ---')
    print(df.dtypes)
    print('Valeurs manquantes :')
    print(df.isna().sum())
    print(f"IDs dupliqués : {df['ID'].duplicated().sum()}")
    print()

## 3 — Distribution par langue (`subset`)
Déjà identifiée comme déséquilibrée (cf. `docs/PROJET.md` / `docs/WORKFLOW.md`) : à confirmer précisément ici avant de décider d'un éventuel rééquilibrage en Phase 2.

In [ ]:
langs = sorted(train['subset'].unique())

dist = pd.DataFrame({
    'train': train['subset'].value_counts().reindex(langs),
    'val':   val['subset'].value_counts().reindex(langs),
    'test':  test['subset'].value_counts().reindex(langs),
})
dist['train_pct'] = (dist['train'] / dist['train'].sum() * 100).round(1)
dist.sort_values('train', ascending=False)

In [ ]:
# Style commun aux graphiques du notebook : marques fines, grille discrète, pas de doubles axes.
plt.rcParams.update({
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'text.color': '#0b0b0b',
    'xtick.color': '#898781',
    'ytick.color': '#898781',
    'axes.grid': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
})

x = range(len(langs))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([i - width/2 for i in x], dist['train'].values, width, label='Train', color='#2a78d6')
ax.bar([i + width/2 for i in x], dist['val'].values, width, label='Val', color='#eb6834')
ax.set_xticks(list(x))
ax.set_xticklabels(langs, rotation=30, ha='right')
ax.set_ylabel('Nombre de lignes')
ax.set_title('Distribution des langues — Train vs Val')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4 — Tokenisation par langue (tokenizer Gemma)
Longueur des questions/réponses en tokens, et efficacité de la tokenisation (tokens/mot) par langue — un ratio élevé signale une faible couverture de la langue dans le pré-entraînement du tokenizer.

⚠️ Nécessite un token Hugging Face avec la licence Gemma acceptée (cf. `docs/WORKFLOW.md`, Phase 0, étapes 9-11).

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

login(token=userdata.get('HF_TOKEN'))

MODEL_NAME = "google/gemma-3n-E2B-it"  # ⚠️ à remplacer par le nom exact du checkpoint vérifié sur HF
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def batch_token_lengths(texts, tokenizer, batch_size=256):
    lengths = []
    texts = [str(t) for t in texts]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, add_special_tokens=False)
        lengths.extend(len(ids) for ids in enc['input_ids'])
    return lengths

train['input_tokens']  = batch_token_lengths(train['input'], tokenizer)
train['output_tokens'] = batch_token_lengths(train['output'], tokenizer)

token_stats = train.groupby('subset')[['input_tokens', 'output_tokens']].agg(['mean', 'median', 'max']).round(1)
token_stats.reindex(langs)

In [ ]:
mean_len = train.groupby('subset')['output_tokens'].mean().reindex(langs)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(langs, mean_len.values, color='#2a78d6')
ax.set_xticklabels(langs, rotation=30, ha='right')
ax.set_ylabel('Longueur moyenne de la réponse (tokens)')
ax.set_title('Longueur moyenne des réponses par langue (Train)')
plt.tight_layout()
plt.show()

In [ ]:
# Tokens/mot : un ratio élevé = tokenizer qui fragmente beaucoup la langue (peu vue au pré-entraînement).
train['output_words'] = train['output'].str.split().str.len().clip(lower=1)
train['tokens_per_word'] = train['output_tokens'] / train['output_words']

tok_ratio = train.groupby('subset')['tokens_per_word'].mean().reindex(langs).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(tok_ratio.index, tok_ratio.values, color='#2a78d6')
ax.set_xticklabels(tok_ratio.index, rotation=30, ha='right')
ax.set_ylabel('Tokens / mot (moyenne)')
ax.set_title("Efficacité de la tokenisation Gemma par langue")
plt.tight_layout()
plt.show()

tok_ratio

## 5 — Réponses dupliquées / "template"
Repère les réponses très courtes ou très répétées (ex. "Yes"/"No"/"Ndiyo"/"Hapana") qui pourraient fausser l'entraînement si elles sont sur-représentées.

In [ ]:
dup_count = train['output'].duplicated().sum()
print(f"Réponses dupliquées dans Train : {dup_count} ({dup_count / len(train):.1%})")

print('\nTop 5 réponses les plus fréquentes par langue :')
top_answers = (
    train.groupby('subset')['output']
    .value_counts()
    .groupby(level=0, group_keys=False)
    .head(5)
)
top_answers

## 6 — Synthèse et décisions pour la Phase 2

À compléter après lecture des résultats ci-dessus :

- **Déséquilibre des langues** : ampleur exacte confirmée en section 3 → décider d'un sur-échantillonnage / pondération de la perte par langue.
- **Longueur des séquences** : `MAX_INPUT_LENGTH` / `MAX_OUTPUT_LENGTH` à fixer en Phase 2-3 à partir des percentiles observés en section 4 (éviter de tronquer trop de réponses, surtout pour les langues aux séquences plus longues).
- **Efficacité de tokenisation** : langues avec un ratio tokens/mot élevé (section 4) demanderont probablement plus d'epochs ou un `max_new_tokens` plus généreux à l'inférence.
- **Réponses "template"** : décider si les réponses très courtes/dupliquées identifiées en section 5 doivent être filtrées, sous-échantillonnées, ou conservées telles quelles.